In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

# Define TPCH orders schema
orders_schema = StructType([
    StructField("o_orderkey", IntegerType(), False),
    StructField("o_custkey", IntegerType(), False),
    StructField("o_orderstatus", StringType(), False),
    StructField("o_totalprice", DoubleType(), False),
    StructField("o_orderdate", DateType(), False),
    StructField("o_orderpriority", StringType(), False),
    StructField("o_clerk", StringType(), False),
    StructField("o_shippriority", IntegerType(), False),
    StructField("o_comment", StringType(), False)
])
##Leemos el csv
df = spark.read.format("csv").option("header", "false").option("delimiter", "|").schema(orders_schema).load("/databricks-datasets/tpch/data-001/orders/")
df.show(5)

##Guardamos en una delta table
df.write.format("delta").mode("overwrite").saveAsTable("orders_raw")

+----------+---------+-------------+------------+-----------+---------------+---------------+--------------+--------------------+
|o_orderkey|o_custkey|o_orderstatus|o_totalprice|o_orderdate|o_orderpriority|        o_clerk|o_shippriority|           o_comment|
+----------+---------+-------------+------------+-----------+---------------+---------------+--------------+--------------------+
|         1|   184501|            O|   203010.51| 1996-01-02|          5-LOW|Clerk#000004753|             0|nstructions sleep...|
|         2|   390010|            O|    75004.81| 1996-12-01|       1-URGENT|Clerk#000004396|             0| foxes. pending a...|
|         3|   616570|            F|   228570.52| 1993-10-14|          5-LOW|Clerk#000004772|             0|sly final account...|
|         4|   683881|            O|     35015.5| 1995-10-11|          5-LOW|Clerk#000000617|             0|sits. slyly regul...|
|         5|   222424|            F|   151840.11| 1994-07-30|          5-LOW|Clerk#0000046

In [0]:
%sql
--CREAMOS DELTA TABLE CLEANED A PARTIR DE RAW, CON ALGUN FILTRO, EL PRECIO DEBE SER MAYOR A 0 Y PEDIDOS ABIERTO, COMPLETADO, PENDIENTE
CREATE OR REPLACE TABLE orders_cleaned AS
SELECT
    o_orderkey,
    o_custkey,
    o_orderstatus,
    o_totalprice,
    o_orderdate,
    o_orderpriority,
    o_clerk,
    o_shippriority,
    o_comment
FROM orders_raw
WHERE o_totalprice > 0
  AND o_orderstatus IN ('F', 'O', 'P');

SELECT * FROM orders_cleaned LIMIT 10

o_orderkey,o_custkey,o_orderstatus,o_totalprice,o_orderdate,o_orderpriority,o_clerk,o_shippriority,o_comment
22522016,710332,F,269633.63,1992-05-10,1-URGENT,Clerk#000000566,0,osits nag pending packages. slyly ironic requests hang carefully. f
22522017,45070,O,293996.52,1997-06-06,1-URGENT,Clerk#000003750,0,nstructions wake besides the blithely
22522018,711257,O,180154.43,1997-06-26,3-MEDIUM,Clerk#000002891,0,"atelets-- ironic, ironic pinto beans are deposit"
22522019,595864,F,148605.91,1992-05-18,4-NOT SPECIFIED,Clerk#000000816,0,riously bold dependencies
22522020,659200,O,211181.28,1995-10-10,5-LOW,Clerk#000001079,0,ideas. slyly silent excuses breach blithely regular theodolites. packages
22522021,164227,P,197075.54,1995-04-23,1-URGENT,Clerk#000001378,0,"s? quickly regular deposits haggle blithely unusual, final theodolites."
22522022,161092,O,128999.68,1998-01-22,2-HIGH,Clerk#000002462,0,y ironic pinto beans: carefully even requests must are along the slyly silent
22522023,696209,O,39315.92,1996-08-24,2-HIGH,Clerk#000003438,0,ly about the furiously re
22522048,426127,O,170105.54,1997-07-03,5-LOW,Clerk#000002369,0,"express, dogged deposits about the furiously unusual accounts"
22522049,137228,P,119605.59,1995-03-25,3-MEDIUM,Clerk#000001918,0,around the carefully express accounts: quic


In [0]:
##CREAMOS DELTA TABLE RAW PARA LINEITEM
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

lineitem_schema = StructType([
    StructField("l_orderkey", IntegerType(), False),
    StructField("l_partkey", IntegerType(), False),
    StructField("l_suppkey", IntegerType(), False),
    StructField("l_linenumber", IntegerType(), False),
    StructField("l_quantity", DoubleType(), False),
    StructField("l_extendedprice", DoubleType(), False),
    StructField("l_discount", DoubleType(), False),
    StructField("l_tax", DoubleType(), False),
    StructField("l_returnflag", StringType(), False),
    StructField("l_linestatus", StringType(), False),
    StructField("l_shipdate", DateType(), False),
    StructField("l_commitdate", DateType(), False),
    StructField("l_receiptdate", DateType(), False),
    StructField("l_shipinstruct", StringType(), False),
    StructField("l_shipmode", StringType(), False),
    StructField("l_comment", StringType(), False)
])

df_lineitem = spark.read.format("csv") \
    .option("header", "false") \
    .option("delimiter", "|") \
    .schema(lineitem_schema) \
    .load("/databricks-datasets/tpch/data-001/lineitem/")

df_lineitem.show(5);

df_lineitem.write.format("delta").mode("overwrite").saveAsTable("lineitem_raw")

+----------+---------+---------+------------+----------+---------------+----------+-----+------------+------------+----------+------------+-------------+-----------------+----------+--------------------+
|l_orderkey|l_partkey|l_suppkey|l_linenumber|l_quantity|l_extendedprice|l_discount|l_tax|l_returnflag|l_linestatus|l_shipdate|l_commitdate|l_receiptdate|   l_shipinstruct|l_shipmode|           l_comment|
+----------+---------+---------+------------+----------+---------------+----------+-----+------------+------------+----------+------------+-------------+-----------------+----------+--------------------+
|         1|   775947|    38463|           1|      17.0|       34389.47|      0.04| 0.02|           N|           O|1996-03-13|  1996-02-12|   1996-03-22|DELIVER IN PERSON|     TRUCK|egular courts abo...|
|         1|   336546|    36547|           2|      36.0|       56971.08|      0.09| 0.06|           N|           O|1996-04-12|  1996-02-28|   1996-04-20| TAKE BACK RETURN|      MAIL|ly

In [0]:
%sql
--CREAMOS LINEITEM CLEANED, QUITAMOS LINEAS CON CANTIDAD 0, PRECIO 0, DESCUENTOS NEGATIVOS E IMPUESTOS NEGATIVOS
CREATE OR REPLACE TABLE lineitem_cleaned AS
SELECT
    l_orderkey,
    l_partkey,
    l_suppkey,
    l_linenumber,
    l_quantity,
    l_extendedprice,
    l_discount,
    l_tax,
    l_returnflag,
    l_linestatus,
    l_shipdate,
    l_commitdate,
    l_receiptdate,
    l_shipinstruct,
    l_shipmode,
    l_comment
FROM lineitem_raw
WHERE l_quantity > 0
  AND l_extendedprice > 0
  AND l_discount >= 0
  AND l_tax >= 0;

  SELECT * FROM lineitem_cleaned LIMIT 10;

l_orderkey,l_partkey,l_suppkey,l_linenumber,l_quantity,l_extendedprice,l_discount,l_tax,l_returnflag,l_linestatus,l_shipdate,l_commitdate,l_receiptdate,l_shipinstruct,l_shipmode,l_comment
16717511,663440,38467,4,39.0,54732.99,0.01,0.07,N,O,1995-09-03,1995-11-01,1995-09-22,DELIVER IN PERSON,AIR,ironic excuses. quickly
16717511,428389,3406,5,28.0,36886.08,0.06,0.0,N,O,1995-11-29,1995-11-11,1995-12-24,NONE,FOB,". regular, regula"
16717511,925737,13292,6,25.0,44067.25,0.04,0.06,N,O,1995-10-24,1995-10-15,1995-10-29,TAKE BACK RETURN,SHIP,es. blithely final deposits use furiousl
16717536,717912,30427,1,27.0,52106.76,0.07,0.01,R,F,1994-08-10,1994-08-03,1994-08-26,COLLECT COD,RAIL,theodolites are about the en
16717536,933844,8881,2,44.0,82623.2,0.05,0.08,R,F,1994-05-08,1994-06-28,1994-05-16,DELIVER IN PERSON,TRUCK,totes use slyly. deposits eat.
16717536,110107,10108,3,3.0,3351.3,0.01,0.06,R,F,1994-08-08,1994-08-02,1994-08-09,TAKE BACK RETURN,FOB,y deposits. ironic requests
16717536,557526,32549,4,9.0,14251.5,0.0,0.07,R,F,1994-08-04,1994-06-18,1994-08-14,NONE,REG AIR,e furiously. regular packages
16717536,335375,10388,5,8.0,11282.88,0.08,0.01,A,F,1994-05-18,1994-06-25,1994-06-15,DELIVER IN PERSON,TRUCK,"lly. regular, pending"
16717536,645926,45927,6,31.0,58028.59,0.03,0.02,R,F,1994-09-05,1994-07-15,1994-10-02,NONE,SHIP,", ironic foxes doze past the asym"
16717536,74861,37363,7,13.0,23866.18,0.02,0.03,R,F,1994-08-04,1994-06-22,1994-09-02,TAKE BACK RETURN,FOB,osits cajole slyly about the sp


In [0]:
%sql
    
--AHORA VAMOS A CREAR UNA TABLA LLAMADA FACT_ORDERS QUE UNIRÁ AMBAS
CREATE OR REPLACE TABLE fact_orders AS
SELECT
    l.l_orderkey,
    l.l_partkey,
    l.l_suppkey,
    l.l_linenumber,
    l.l_quantity,
    l.l_extendedprice,
    l.l_discount,
    l.l_tax,
    l.l_returnflag,
    l.l_linestatus,
    l.l_shipdate,
    l.l_commitdate,
    l.l_receiptdate,
    l.l_shipinstruct,
    l.l_shipmode,
    l.l_comment,
    o.o_custkey,
    o.o_orderstatus,
    o.o_totalprice,
    o.o_orderdate,
    o.o_orderpriority,
    o.o_clerk,
    o.o_shippriority,
    o.o_comment AS order_comment
FROM lineitem_cleaned l
JOIN orders_cleaned o
  ON l.l_orderkey = o.o_orderkey;

SELECT * FROM fact_orders 
ORDER BY l_orderkey, l_linenumber
LIMIT 10;

l_orderkey,l_partkey,l_suppkey,l_linenumber,l_quantity,l_extendedprice,l_discount,l_tax,l_returnflag,l_linestatus,l_shipdate,l_commitdate,l_receiptdate,l_shipinstruct,l_shipmode,l_comment,o_custkey,o_orderstatus,o_totalprice,o_orderdate,o_orderpriority,o_clerk,o_shippriority,order_comment
1,775947,38463,1,17.0,34389.47,0.04,0.02,N,O,1996-03-13,1996-02-12,1996-03-22,DELIVER IN PERSON,TRUCK,egular courts above the,184501,O,203010.51,1996-01-02,5-LOW,Clerk#000004753,0,nstructions sleep furiously among
1,336546,36547,2,36.0,56971.08,0.09,0.06,N,O,1996-04-12,1996-02-28,1996-04-20,TAKE BACK RETURN,MAIL,ly final dependencies: slyly bold,184501,O,203010.51,1996-01-02,5-LOW,Clerk#000004753,0,nstructions sleep furiously among
1,318499,18500,3,8.0,12139.84,0.1,0.02,N,O,1996-01-29,1996-03-05,1996-01-31,TAKE BACK RETURN,REG AIR,"riously. regular, express dep",184501,O,203010.51,1996-01-02,5-LOW,Clerk#000004753,0,nstructions sleep furiously among
1,10658,23159,4,28.0,43922.2,0.09,0.06,N,O,1996-04-21,1996-03-30,1996-05-16,NONE,AIR,lites. fluffily even de,184501,O,203010.51,1996-01-02,5-LOW,Clerk#000004753,0,nstructions sleep furiously among
1,120134,7641,5,24.0,27699.12,0.1,0.04,N,O,1996-03-30,1996-03-14,1996-04-01,NONE,FOB,pending foxes. slyly re,184501,O,203010.51,1996-01-02,5-LOW,Clerk#000004753,0,nstructions sleep furiously among
1,78173,3176,6,32.0,36837.44,0.07,0.02,N,O,1996-01-30,1996-02-07,1996-02-03,DELIVER IN PERSON,MAIL,arefully slyly ex,184501,O,203010.51,1996-01-02,5-LOW,Clerk#000004753,0,nstructions sleep furiously among
2,530849,5870,1,38.0,71433.16,0.0,0.05,N,O,1997-01-28,1997-01-14,1997-02-02,TAKE BACK RETURN,RAIL,ven requests. deposits breach a,390010,O,75004.81,1996-12-01,1-URGENT,Clerk#000004396,0,"foxes. pending accounts at the pending, silent asymptot"
3,21485,8986,1,45.0,63291.6,0.06,0.0,R,F,1994-02-02,1994-01-04,1994-02-23,NONE,AIR,ongside of the furiously brave acco,616570,F,228570.52,1993-10-14,5-LOW,Clerk#000004772,0,sly final accounts boost. carefully regular ideas cajole carefully. depos
3,95178,32682,2,49.0,57485.33,0.1,0.0,R,F,1993-11-09,1993-12-20,1993-11-24,TAKE BACK RETURN,RAIL,unusual accounts. eve,616570,F,228570.52,1993-10-14,5-LOW,Clerk#000004772,0,sly final accounts boost. carefully regular ideas cajole carefully. depos
3,642242,17267,3,27.0,31973.67,0.06,0.07,A,F,1994-01-16,1993-11-22,1994-01-23,DELIVER IN PERSON,SHIP,nal foxes wake.,616570,F,228570.52,1993-10-14,5-LOW,Clerk#000004772,0,sly final accounts boost. carefully regular ideas cajole carefully. depos


In [0]:
%sql
    
--CREAMOS DIMDATE
SELECT
  MIN(date) AS min_date,
  MAX(date) AS max_date
FROM (
  SELECT o_orderdate AS date FROM orders_cleaned
  UNION ALL
  SELECT l_shipdate FROM lineitem_cleaned
  UNION ALL
  SELECT l_commitdate FROM lineitem_cleaned
  UNION ALL
  SELECT l_receiptdate FROM lineitem_cleaned
);

--CREAMOS LA TABLA DE FECHAS
CREATE OR REPLACE TABLE dim_date AS
WITH date_range AS (
  SELECT
    sequence(
      (SELECT MIN(o_orderdate) FROM orders_cleaned),
      (SELECT MAX(l_receiptdate) FROM lineitem_cleaned),
      interval 1 day
    ) AS dates
)
SELECT
  explode(dates) AS date,
  year(date) AS year,
  quarter(date) AS quarter,
  month(date) AS month,
  day(date) AS day,
  weekofyear(date) AS week,
  date_format(date, 'MMMM') AS month_name,
  date_format(date, 'EEEE') AS day_name
FROM date_range;

SELECT * FROM dim_date ORDER BY date LIMIT 20;

date,year,quarter,month,day,week,month_name,day_name
1992-01-01,1992,1,1,1,1,January,Wednesday
1992-01-02,1992,1,1,2,1,January,Thursday
1992-01-03,1992,1,1,3,1,January,Friday
1992-01-04,1992,1,1,4,1,January,Saturday
1992-01-05,1992,1,1,5,1,January,Sunday
1992-01-06,1992,1,1,6,2,January,Monday
1992-01-07,1992,1,1,7,2,January,Tuesday
1992-01-08,1992,1,1,8,2,January,Wednesday
1992-01-09,1992,1,1,9,2,January,Thursday
1992-01-10,1992,1,1,10,2,January,Friday


In [0]:
##CREAMOS TABLA CUSTOMER
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

customer_schema = StructType([
    StructField("c_custkey", IntegerType(), False),
    StructField("c_name", StringType(), False),
    StructField("c_address", StringType(), False),
    StructField("c_nationkey", IntegerType(), False),
    StructField("c_phone", StringType(), False),
    StructField("c_acctbal", DoubleType(), False),
    StructField("c_mktsegment", StringType(), False),
    StructField("c_comment", StringType(), False)
])
#leemos el dataset
df_customer = spark.read.format("csv") \
    .option("header", "false") \
    .option("delimiter", "|") \
    .schema(customer_schema) \
    .load("/databricks-datasets/tpch/data-001/customer/")

df_customer.show(5)

#CREAMOS LA DELTA TABLE CUSTOMER_RAW

df_customer.write.format("delta").mode("overwrite").saveAsTable("customer_raw")

+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+
|c_custkey|            c_name|           c_address|c_nationkey|        c_phone|c_acctbal|c_mktsegment|           c_comment|
+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+
|        1|Customer#000000001|   IVhzIApeRb ot,c,E|         15|25-989-741-2988|   711.56|    BUILDING|to the even, regu...|
|        2|Customer#000000002|XSTf4,NCwDVaWNe6t...|         13|23-768-687-3665|   121.65|  AUTOMOBILE|l accounts. blith...|
|        3|Customer#000000003|        MG9kdTD2WBHm|          1|11-719-748-3364|  7498.12|  AUTOMOBILE| deposits eat sly...|
|        4|Customer#000000004|         XxVSJsLAGtn|          4|14-128-190-5944|  2866.83|   MACHINERY| requests. final,...|
|        5|Customer#000000005|KvpyuHCplrB84WgAi...|          3|13-750-942-6364|   794.47|   HOUSEHOLD|n accounts will h...|
+-------

In [0]:
%sql
--CREAMOS CLEANED DE CUSTOMER
CREATE OR REPLACE TABLE dim_customer AS
SELECT
    c_custkey,
    c_name,
    c_address,
    c_nationkey,
    c_phone,
    c_acctbal,
    c_mktsegment,
    c_comment
FROM customer_raw;

num_affected_rows,num_inserted_rows


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
##CREAMOS RAW TABLE DE SUPPLIER
supplier_schema = StructType([
    StructField("s_suppkey", IntegerType(), False),
    StructField("s_name", StringType(), False),
    StructField("s_address", StringType(), False),
    StructField("s_nationkey", IntegerType(), False),
    StructField("s_phone", StringType(), False),
    StructField("s_acctbal", DoubleType(), False),
    StructField("s_comment", StringType(), False)
])

df_supplier = spark.read.format("csv") \
    .option("header", "false") \
    .option("delimiter", "|") \
    .schema(supplier_schema) \
    .load("/databricks-datasets/tpch/data-001/supplier/")

df_supplier.show(5)

##CREAMOS DELTATABLE RAW PARA SUPPLIER
df_supplier.write.format("delta").mode("overwrite").saveAsTable("supplier_raw")

+---------+------------------+--------------------+-----------+---------------+---------+--------------------+
|s_suppkey|            s_name|           s_address|s_nationkey|        s_phone|s_acctbal|           s_comment|
+---------+------------------+--------------------+-----------+---------------+---------+--------------------+
|        1|Supplier#000000001| N kD4on9OM Ipw3,...|         17|27-918-335-1736|  5755.94|each slyly above ...|
|        2|Supplier#000000002|89eJ5ksX3ImxJQBvx...|          5|15-679-861-2259|  4032.68| slyly bold instr...|
|        3|Supplier#000000003|q1,G3Pj6OjIuUYfUo...|          1|11-383-516-1199|   4192.4|blithely silent r...|
|        4|Supplier#000000004|Bk7ah4CK8SYQTepEm...|         15|25-843-787-7479|  4641.08|riously even requ...|
|        5|Supplier#000000005|   Gcdm2rJRzl5qlTVzc|         11|21-151-690-3663|  -283.84|. slyly regular p...|
+---------+------------------+--------------------+-----------+---------------+---------+--------------------+
o

In [0]:
%sql
--CREAMOS LA CLEANED DELTATABLE DE SUPPLIER
CREATE OR REPLACE TABLE dim_supplier AS
SELECT
    s_suppkey,
    s_name,
    s_address,
    s_nationkey,
    s_phone,
    s_acctbal,
    s_comment
FROM supplier_raw;


num_affected_rows,num_inserted_rows
